# Notebook 2 — GitHub Webhook Handler

**What you'll learn:**
1. How GitHub webhooks work
2. How to verify webhook signatures (security)
3. How to parse a `pull_request` event payload
4. How to run and expose the Flask server locally with ngrok

## How GitHub Webhooks Work

```
Developer opens PR
       │
       ▼
   GitHub.com
       │  HTTP POST (JSON payload)
       ▼
Your Flask server  (/webhook)
       │
       ├── 1. Verify HMAC-SHA256 signature
       ├── 2. Parse pull_request event
       ├── 3. Fetch the PR diff
       ├── 4. Call AI reviewer
       └── 5. Post review comment back to GitHub
```

GitHub signs every request with a shared secret so you can verify it actually came from GitHub.

## Understanding Signature Verification

In [ ]:
import hmac, hashlib, json

# GitHub signs the raw payload bytes with your webhook secret
SECRET  = b'my-webhook-secret'
PAYLOAD = json.dumps({'action': 'opened', 'pull_request': {'number': 42}}).encode()

# GitHub sends this header: X-Hub-Signature-256
signature = 'sha256=' + hmac.new(SECRET, PAYLOAD, hashlib.sha256).hexdigest()
print('Signature GitHub sends:', signature)

# You verify it on receipt:
def verify(payload_bytes, received_sig, secret):
    expected = 'sha256=' + hmac.new(secret.encode(), payload_bytes, hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected, received_sig)

print('Valid signature:', verify(PAYLOAD, signature, 'my-webhook-secret'))
print('Tampered check :', verify(PAYLOAD, signature, 'wrong-secret'))

## Parsing a pull_request Event Payload

This is a simplified version of what GitHub sends.

In [ ]:
SAMPLE_PAYLOAD = {
    'action': 'opened',
    'pull_request': {
        'number': 42,
        'title': 'Fix SQL injection in login route',
        'body': 'This PR fixes the login endpoint to use parameterised queries.',
        'head': {'sha': 'abc123def456'},
        'user': {'login': 'dev-alice'},
    },
    'repository': {
        'name': 'my-app',
        'owner': {'login': 'my-org'},
    },
}

pr     = SAMPLE_PAYLOAD['pull_request']
repo   = SAMPLE_PAYLOAD['repository']
action = SAMPLE_PAYLOAD['action']

print(f"Action     : {action}")
print(f"PR Number  : {pr['number']}")
print(f"PR Title   : {pr['title']}")
print(f"Head SHA   : {pr['head']['sha']}")
print(f"Repo       : {repo['owner']['login']}/{repo['name']}")
print(f"Author     : {pr['user']['login']}")

# Only review on these actions
TRIGGER_ACTIONS = ('opened', 'synchronize', 'reopened')
should_review = action in TRIGGER_ACTIONS
print(f"\nShould trigger review: {should_review}")

## Run the Flask Server

Open a **terminal** in the project root and run:

```bash
# Set env vars first
export FLASK_APP=app.webhook_handler
export FLASK_ENV=development

# Start the server
python -m flask run --port 5001
```

Test it's alive:

In [ ]:
import requests

try:
    r = requests.get('http://localhost:5001/health', timeout=2)
    print('Server response:', r.json())
except requests.ConnectionError:
    print('Server not running — start it in a terminal first (see above).')

## Expose Locally with ngrok

GitHub needs a public URL to send webhooks to. ngrok creates a secure tunnel.

**Option A — Terminal (recommended):**
```bash
ngrok http 5001
# Copy the https://xxxxx.ngrok-free.app URL
```

**Option B — Python:**

In [ ]:
# Uncomment to run (requires pyngrok and a running Flask server)
# from pyngrok import ngrok
# tunnel = ngrok.connect(5001)
# print('Public URL:', tunnel.public_url)
# print('Webhook URL:', tunnel.public_url + '/webhook')

print('Steps to set up the GitHub webhook:')
print('1. Go to: https://github.com/<your-org>/<your-repo>/settings/hooks')
print('2. Click "Add webhook"')
print('3. Payload URL: https://<ngrok-url>/webhook')
print('4. Content type: application/json')
print('5. Secret: (same as GITHUB_WEBHOOK_SECRET in your .env)')
print('6. Events: select "Pull requests" only')
print('7. Click "Add webhook"')

---
**Next:** Open `03_ai_review_engine.ipynb` to see how the AI analysis works.